# 🎯 [MNT04_M01] 헬기/기갑 엔진 및 구동계 진동, 온도, 압력 센서 시계열 데이터셋 정의
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT04/mnt04_m01_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT04_M01 |
| **미션 주제** | 헬기/기갑 엔진 및 구동계 진동, 온도, 압력 센서 시계열 데이터셋 정의 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
항공·기갑 장비의 엔진과 구동계는 고장이 나기 전에 진동, 온도, 압력에 미세한 변화가 먼저 나타나지만, 고정 임계값만으로는 고부하 운용일을 이상으로 오인하거나 초기 열화를 놓쳐 수리부속을 제때 확보하지 못한다. 본 프로젝트는 고장 실적과 수리부속 소모를 연결해 정비와 부품 관리의 연계 문제를 학습하는 것을 목표로, 교육용 가상 기동체 48대의 센서 시계열과 부품 소모를 코드로 만들어 분석한다. 센서 데이터셋 정의, 잡음 제거와 FFT 특징량 추출, 임계값 Baseline과 부품 소모 파레토 분석, 오토인코더 기반 미세 변화 감지와 파생 피처 모델, 잔존수명(RUL) 예측과 이상 신뢰도 점수, 설명가능한 경보 화면, 검사관 승인(HITL) 기반 조기 입고 시뮬레이터, 대용량 스트림 스트레스 시험, AAR 오경보 분석과 임계 재보정, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 고장 전에 이상 징후를 일찍 잡아 정비 시점과 수리부속 소요를 예고하되, 조기 입고 결정은 검사관이 근거를 보고 내리도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 정비 야전(정비대대, 군수지원사령부)에 즉시 도입 가능한 "완성형 국방 예지정비(PdM) 및 PBL 품목관리 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 도메인 및 데이터 규격서** | 군수 도메인 분석서 및 데이터 딕셔너리(`docs/mnt04_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mnt04_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **센서 데이터 정제 및 전처리 모듈** | 고주파 센서 신호 잡음 제거, 결측치 보정 및 FFT 특징 추출 파이프라인(`src/etl/mnt04_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **탐색적 데이터 분석(EDA) & Baseline** | 3-시그마 임계값 Baseline 및 부품 소모 파레토 분석(`notebooks/mnt04_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 오토인코더 기반 미세 진동 변화 감지 및 파생 피처 엔지니어링 모듈(`src/features/mnt04_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 잔존수명(RUL) 엔진** | 부품 RUL 예측 앙상블 모델 및 종합 상태 이상 신뢰도 엔진(`src/models/mnt04_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | 오토인코더 재구성 오차 분해 및 XAI 원인 설명기(`src/explainability/mnt04_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **HITL 조기 입고 시뮬레이터** | 검사관 HITL 승인 큐, 감사 추적 및 조기 정비 시뮬레이터(`src/optimization/mnt04_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **스트림 주입 & 관제 대시보드** | 대용량 센서 스트림 주입 스트레스 시험 및 실시간 관제 대시보드(`src/dashboard/mnt04_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 오경보 원인 분석 및 임계 파라미터 재보정 ROI 보고서(`reports/mnt04_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 데이터 기반 챔피언-도전자 자동 재학습 및 E2E 통합 파이프라인(`pipeline/mnt04_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 헬기/기갑 엔진 및 구동계 진동, 온도, 압력 센서 시계열 데이터셋 정의
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

헬기/기갑 엔진 진동, 온도, 압력 센서 시계열 데이터셋을 정의한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **헬기/기갑 엔진 및 구동계의 진동, 온도, 압력 센서 시계열과 고장·수리부속 소모 실적의 데이터셋 스키마, 데이터 딕셔너리, 업무 프로세스 맵을 정의한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **센서 수집부터 전처리, 특징 추출, 이상 감지, RUL 예측, 검사관 승인, 정비·부품 소모, 사후검토까지 업무 흐름을 단계별로 정리하고, 센서·특징·고장·부품 묶음으로 피처 20개 이상 정의하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/mnt04_logistics_domain_spec.md 및 데이터 딕셔너리` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 실무 업무 프로세스와의 일치성 및 분석 목표의 명확성

#### 미션 PASS 조건

- 정량 기준: 업무 프로세스 맵 작성 완료, 필요 피처(Feature) 목록 20개 이상 정의


- 진행 기준: 도메인 분석서 및 데이터 요구명세서 승인


- 검증 방법: 군수 담당 실무자 인터뷰 및 기존 전산체계 데이터 구조 대조


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# ============================================================================
# [미션 01 산출물 파일 연계]: docs/mnt04_logistics_domain_spec.md (및 데이터 딕셔너리)
# (선택) 로컬 모듈 파일로 추출 저장 시: %%writefile docs/mnt04_logistics_domain_spec.md
# ============================================================================
"""헬기/기갑 엔진 및 구동계의 진동·온도·압력 센서 시계열, 고장 실적, 수리부속 소모 기록의 데이터셋 스키마·데이터 딕셔너리(피처 20개 이상)와 예지정비·PBL 품목관리 업무 프로세스 맵을 정의하고, 가상 센서 자료로 검증한다."""
import json
import re
import numpy as np
import pandas as pd
from scipy.stats import kurtosis

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 기동체 48대의 엔진·구동계 진동·온도·압력 센서 자료 생성기 (실제 장비·부대·정비 실적 없음)
# 개체마다 정상 -> 열화 -> 고장(수명종료)까지 하루 1회 측정(진동 파형 256표본 + 온도 + 압력 + 부하)을 만든다.
# 열화 유형 4가지: bearing(베어링), gear(기어), oil_pump(오일 펌프), cooling(냉각 팬 모터)
# ---------------------------------------------------------------------------
FS, NS = 1024, 256                              # 가상 표본화 주파수(Hz), 파형 1개당 표본 수
TT = np.arange(NS) / FS
MODES = {  # 열화 유형: (발생 확률, 평균 수명(일), 교체 수리부속, 가상 단가지수)
    "bearing": (0.40, 160, "주 베어링 세트", 6.0), "gear": (0.25, 175, "구동 기어 세트", 9.0),
    "oil_pump": (0.20, 150, "오일 펌프", 4.0), "cooling": (0.15, 140, "냉각 팬 모터", 3.0)}
KIT_COST = 1.0                                  # 모든 정비에 함께 소모되는 씰·필터 키트 단가지수
BANDS = {"b50": (40, 60), "b100": (90, 110), "b150": (140, 156), "b165": (156, 176), "b200": (190, 215)}


def make_runs(n=48, seed=1):
    """반환: units(개체 정보), snap(일별 측정 요약), W(진동 파형 [측정 수, 256])"""
    rng = np.random.default_rng(seed)
    names = list(MODES)
    mode = rng.choice(names, n, p=[MODES[m][0] for m in names])
    life = np.clip(np.array([rng.normal(MODES[m][1], 18) for m in mode]), 90, 230).astype(int)
    start = np.sort(rng.integers(0, 240, n))
    units = pd.DataFrame({"unit": [f"U{i:02d}" for i in range(n)], "mode": mode, "life": life, "start": start, "base_load": rng.uniform(0.4, 0.75, n)})
    rows = []
    for u in units.itertuples():
        d = np.arange(u.life + 1)
        onset = 0.45 * u.life
        deg = np.clip((d - onset) / (u.life - onset), 0, 1) ** 1.5
        load = np.clip(u.base_load + rng.normal(0, 0.06, len(d)), 0.2, 1.0)
        hi = rng.random(len(d)) < 0.07                                            # 고부하 운용일(정상이지만 진동·온도 상승)
        load = np.where(hi, rng.uniform(0.95, 1.1, len(d)), load)
        rows.append(pd.DataFrame({"unit": u.unit, "day": d, "cal": u.start + d, "rul": u.life - d, "deg": deg, "load": load, "hi_load": hi, "mode": u.mode}))
    snap = pd.concat(rows, ignore_index=True)
    S = len(snap)
    snap["ambient"] = 15 + 12 * np.sin(2 * np.pi * snap["cal"] / 365) + rng.normal(0, 2, S)
    dg, ld, md = snap["deg"].values, snap["load"].values, snap["mode"].values
    isb, isg, iso, isc = (md == "bearing"), (md == "gear"), (md == "oil_pump"), (md == "cooling")
    snap["temp"] = 55 + 22 * ld + 0.5 * snap["ambient"] + 9 * dg * iso + 26 * dg * isc + 3 * dg * (isb | isg) + rng.normal(0, 1.5, S)
    snap["press"] = 450 - 12 * ld - 85 * dg * iso - 15 * dg * isc + rng.normal(0, 6, S)
    ph = rng.uniform(0, 2 * np.pi, (S, 3))
    a1, a2 = 1.0 * (0.5 + ld), 0.4 * (0.5 + ld)
    W = (a1[:, None] * np.sin(2 * np.pi * 50 * TT + ph[:, :1]) + a2[:, None] * np.sin(2 * np.pi * 100 * TT + ph[:, 1:2])
         + rng.normal(0, 0.35, (S, NS)))
    W += (0.9 * dg * isb)[:, None] * np.sin(2 * np.pi * 165 * TT + ph[:, 2:3])
    W += (0.8 * dg * isg)[:, None] * (np.sin(2 * np.pi * 150 * TT + ph[:, 2:3]) + 0.8 * np.sin(2 * np.pi * 100 * TT))
    W += (0.5 * dg * iso)[:, None] * np.sin(2 * np.pi * 200 * TT + ph[:, 2:3])
    k = rng.poisson(10 * dg * isb)                                                 # 베어링 열화: 충격성 펄스(첨도 상승)
    pos, amp = rng.integers(0, NS, (S, 30)), rng.normal(0, 3.0, (S, 30)) * dg[:, None]
    amp = amp * (np.arange(30)[None, :] < k[:, None])
    np.add.at(W, (np.repeat(np.arange(S), 30), pos.ravel()), amp.ravel())
    return units, snap, W


def extract(W):
    """진동 파형 -> 특징량: RMS, 첨도, 파고율, 최대-최소, FFT 대역 에너지(50/100/150/165/200Hz 부근)"""
    x = W - W.mean(axis=1, keepdims=True)
    rms = np.sqrt((x ** 2).mean(axis=1))
    out = {"rms": rms, "kurt": kurtosis(x, axis=1, fisher=False), "crest": np.abs(x).max(axis=1) / rms, "p2p": x.max(axis=1) - x.min(axis=1)}
    A = np.abs(np.fft.rfft(x * np.hanning(NS), axis=1)) / NS
    f = np.fft.rfftfreq(NS, 1 / FS)
    for nm, (lo, hi) in BANDS.items():
        out[nm] = np.sqrt((A[:, (f >= lo) & (f < hi)] ** 2).sum(axis=1))
    return pd.DataFrame(out)

# 1) 업무 프로세스 맵: 센서 수집부터 수리부속 소요 환류까지 (각 단계에서 생기는 데이터)
PROCESS = [("센서 수집", "개체 ID, 측정일, 진동 파형, 온도, 압력, 부하"), ("전처리", "잡음 제거, 단위·표기 통일, 결측 보정"),
           ("특징 추출", "RMS, 첨도, 파고율, FFT 대역 에너지"), ("이상 감지", "이상 점수, 경보 단계"),
           ("잔존수명(RUL) 예측", "고장 임박 시점, 신뢰 구간"), ("검사관 승인", "조기 입고 제안, 승인·반려 사유"),
           ("정비·부품 소모", "교체 부품, 수량, 정비 소요시간"), ("고장 실적 연결", "정비 건 ID로 센서 이력과 고장 실적 연결"),
           ("부품 소요·가동률 집계", "품목별 소모, 가용도(PBL 지표)"), ("사후검토(AAR)", "오경보 원인, 임계 파라미터 조정")]

# 2) 스키마(데이터 딕셔너리): 그룹 / 필드 / 자료형 / 규칙 / 설명
S = {
    "sensor": {
        "unit_id": (str, {"pattern": r"^U\d{2}$"}, "개체 ID(가상 기동체)"),
        "day": (int, {"min": 0, "max": 260}, "운용 경과일(측정일)"),
        "vib_rms": (float, {"min": 0, "max": 10}, "진동 RMS(가상 단위)"),
        "vib_kurt": (float, {"min": 1, "max": 30}, "진동 첨도(충격성 지표)"),
        "temp": (float, {"min": -30, "max": 150}, "엔진 온도(섭씨)"),
        "press": (float, {"min": 0, "max": 700}, "오일 압력(kPa)"),
        "load": (float, {"min": 0, "max": 1.2}, "운용 부하율"),
    },
    "feature": {
        "band_50": (float, {"min": 0}, "50Hz 부근 FFT 대역 에너지"),
        "band_165": (float, {"min": 0}, "165Hz 부근(베어링 결함 대역) 에너지"),
        "crest": (float, {"min": 1, "max": 30}, "파고율(최대값/RMS)"),
        "hf_ratio": (float, {"min": 0}, "고주파/기본주파 에너지 비"),
        "temp_res": (float, {"min": -60, "max": 80}, "부하·외기 대비 온도 잔차"),
    },
    "failure": {
        "event_id": (str, {"pattern": r"^F\d{3}$"}, "고장·정비 건 ID(센서 이력과 연결하는 키)"),
        "mode": (str, {"enum": list(MODES)}, "열화(고장) 유형"),
        "life": (int, {"min": 60, "max": 260}, "고장까지 운용일수"),
        "unplanned": (bool, {}, "비계획 정비 여부"),
    },
    "parts": {
        "part_name": (str, {"enum": [v[2] for v in MODES.values()]}, "소모 수리부속"),
        "qty": (int, {"min": 1, "max": 10}, "소모 수량"),
        "unit_cost": (float, {"min": 0, "max": 50}, "가상 단가지수"),
    },
    "target": {
        "rul": (float, {"min": 0, "max": 260}, "잔존수명(일) - 예측대상 1"),
        "anomaly_score": (float, {"min": 0}, "이상 점수(재구성 오차) - 예측대상 2"),
        "availability": (float, {"min": 0, "max": 1}, "가용도(가동일/전체일) - PBL 성과 지표"),
    },
}


def check(name, v, typ, rule):
    if not isinstance(v, typ) or (typ in (int, float) and isinstance(v, bool)):
        return [f"{name}: 자료형 오류({type(v).__name__})"]
    e = []
    if "enum" in rule and v not in rule["enum"]:
        e.append(f"{name}: 허용되지 않은 값 '{v}'")
    if "min" in rule and v < rule["min"]:
        e.append(f"{name}: 최소값 {rule['min']} 미만")
    if "max" in rule and v > rule["max"]:
        e.append(f"{name}: 최대값 {rule['max']} 초과")
    if "pattern" in rule and not re.match(rule["pattern"], v):
        e.append(f"{name}: 형식 불일치 '{v}'")
    return e


def validate(rec):
    errs = []
    for g, fields in S.items():
        for k, (typ, rule, _) in fields.items():
            if k not in rec.get(g, {}):
                errs.append(f"{g}.{k}: 필수 필드 누락")
            else:
                errs += check(f"{g}.{k}", rec[g][k], typ, rule)
    return errs


# 3) [교육용가상자료] 생성 자료에서 20건을 레코드로 만들어 검증
units, snap, W = make_runs()
F = extract(W)
snap = pd.concat([snap, F], axis=1)
snap["temp_res"] = snap["temp"] - (55 + 22 * snap["load"] + 0.5 * snap["ambient"])
snap["hf_ratio"] = (snap["b150"] + snap["b165"] + snap["b200"]) / (snap["b50"] + 1e-6)
ui = units.set_index("unit")
recs = []
for i, r in enumerate(snap.sample(20, random_state=0).itertuples()):
    m = r.mode
    recs.append({
        "sensor": {"unit_id": r.unit, "day": int(r.day), "vib_rms": float(round(r.rms, 3)), "vib_kurt": float(round(r.kurt, 2)),
                   "temp": float(round(r.temp, 1)), "press": float(round(r.press, 1)), "load": float(round(r.load, 2))},
        "feature": {"band_50": float(round(r.b50, 4)), "band_165": float(round(r.b165, 4)), "crest": float(round(r.crest, 2)),
                    "hf_ratio": float(round(r.hf_ratio, 3)), "temp_res": float(round(r.temp_res, 2))},
        "failure": {"event_id": f"F{int(r.unit[1:]):03d}", "mode": m, "life": int(ui.loc[r.unit, "life"]), "unplanned": bool(r.rul < 5)},
        "parts": {"part_name": MODES[m][2], "qty": 1, "unit_cost": float(MODES[m][3])},
        "target": {"rul": float(r.rul), "anomaly_score": float(round(r.deg, 3)), "availability": 0.92}})
ok = sum(not validate(r) for r in recs)
print(f"가상 센서 자료 {len(snap)}건(개체 {len(units)}대), 표본 {len(recs)}건 중 검증 통과 {ok}건, 오류 {len(recs) - ok}건")
print(f"전체 파형 {W.shape[0]}개 x {W.shape[1]}표본, 온도·압력 결측 {int(snap[['temp', 'press']].isna().sum().sum())}건")

# 4) 오류 주입 시험
bad = json.loads(json.dumps(recs[0]))
bad["sensor"]["unit_id"] = "ENGINE-3"           # ID 형식 오류
bad["sensor"]["temp"] = 480.0                  # 센서 범위 초과(단위 혼동 의심)
bad["failure"]["mode"] = "turbine"             # 정의되지 않은 열화 유형
del bad["parts"]["qty"]                        # 필수 필드 누락
bad["target"]["rul"] = "30일"                  # 자료형 오류
print("\n[오류 주입 결과]")
for e in validate(bad):
    print(" -", e)

# 5) 프로세스 맵과 데이터 딕셔너리
print("\n[업무 프로세스 맵]")
print(" -> ".join(f"{i}.{p}" for i, (p, _) in enumerate(PROCESS, 1)))
for p, d in PROCESS:
    print(f"   {p:12s}: {d}")
rows = [(g, k, t.__name__, d) for g, fl in S.items() for k, (t, r, d) in fl.items()]
dd = pd.DataFrame(rows, columns=["그룹", "필드", "자료형", "설명"])
print(f"\n[데이터 딕셔너리] 피처 {len(dd)}개 (목표 20개 이상: {'달성' if len(dd) >= 20 else '미달'})")
print(dd.to_string(index=False))


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
가상 센서 자료 7733건(개체 48대), 표본 20건 중 검증 통과 20건, 오류 0건
전체 파형 7733개 x 256표본, 온도·압력 결측 0건

[오류 주입 결과]
 - sensor.unit_id: 형식 불일치 'ENGINE-3'
 - sensor.temp: 최대값 150 초과
 - failure.mode: 허용되지 않은 값 'turbine'
 - parts.qty: 필수 필드 누락
 - target.rul: 자료형 오류(str)

[업무 프로세스 맵]
1.센서 수집 -> 2.전처리 -> 3.특징 추출 -> 4.이상 감지 -> 5.잔존수명(RUL) 예측 -> 6.검사관 승인 -> 7.정비·부품 소모 -> 8.고장 실적 연결 -> 9.부품 소요·가동률 집계 -> 10.사후검토(AAR)
   센서 수집       : 개체 ID, 측정일, 진동 파형, 온도, 압력, 부하
   전처리         : 잡음 제거, 단위·표기 통일, 결측 보정
   특징 추출       : RMS, 첨도, 파고율, FFT 대역 에너지
   이상 감지       : 이상 점수, 경보 단계
   잔존수명(RUL) 예측: 고장 임박 시점, 신뢰 구간
   검사관 승인      : 조기 입고 제안, 승인·반려 사유
   정비·부품 소모    : 교체 부품, 수량, 정비 소요시간
   고장 실적 연결    : 정비 건 ID로 센서 이력과 고장 실적 연결
   부품 소요·가동률 집계: 품목별 소모, 가용도(PBL 지표)
   사후검토(AAR)   : 오경보 원인, 임계 파라미터 조정

[데이터 딕셔너리] 피처 22개 (목표 20개 이상: 달성)
     그룹            필드   자료형                        설명
 sensor       unit_id   str             개체 ID(가상 기동체)
 sensor           day   int               운용 경과일(측정일)
 sensor       vib_rms float             진동 RMS(가상 단위)
 sensor      vib_kurt float             진동 첨도(충격성 지표)
 sensor          temp float                 엔진 온도(섭씨)
 sensor         press float                오일 압력(kPa)
 sensor          load float                    운용 부하율
feature       band_50 float        50Hz 부근 FFT 대역 에너지
feature      band_165 float   165Hz 부근(베어링 결함 대역) 에너지
feature         crest float              파고율(최대값/RMS)
feature      hf_ratio float            고주파/기본주파 에너지 비
feature      temp_res float            부하·외기 대비 온도 잔차
failure      event_id   str 고장·정비 건 ID(센서 이력과 연결하는 키)
failure          mode   str                 열화(고장) 유형
failure          life   int                 고장까지 운용일수
... (이하 생략)
```
</details>
